# Семинар 9. Исследуем обученную модель

По двум наборам данных разберём, как изучать важность признаков, форму прогнозной функции и ошибки предиктора. Начнём с готового примера на дневном велопрокате, затем выполним самостоятельное исследование цены бриллиантов.

Рабочие инструменты: `feature_importances_`, `permutation_importance`, `PartialDependenceDisplay`, группировка ошибок. Для каждого метода будем рассматривать его входы, полученный результат и содержательную интерпретацию.

---

# Часть I. Дневной велопрокат: от дерева к исследованию предиктора

**Задача.** По погоде и календарным признакам оценить число прокатов велосипедов за день (`cnt`). Одна строка таблицы — один день в Вашингтоне, 2011–2012 годы.

Данные: [UCI Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), H. Fanaee-T и J. Gama, лицензия CC BY 4.0. Мы используем исторически измеренную погоду; это исследование моделей на известных днях, а не испытание прогноза погоды на будущий год.

## 1. Загрузка и знакомство с данными

Подключим библиотеки. Разделение импортов по назначению поможет видеть, какие API нужны для каждой операции.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

Данные Capital Bikeshare уже подготовлены для анализа: значения температуры, влажности и скорости ветра указаны в физических единицах, категории подписаны понятными названиями. Одна строка — один день.

In [ ]:
bike = pd.read_csv('data/bike_day_prepared.csv', parse_dates=['date'])

In [ ]:
bike.head()

В таблице 730 дней 2011–2012 годов. `cnt` — число прокатов; `temp` — температура в °C, `hum` — влажность в %, `wind` — скорость ветра в км/ч. `season` и `weather` — категории, `holiday` и `workingday` — признаки «да/нет». `year` и `month` понадобятся для разделения данных и диагностики.

In [ ]:
bike.shape

Проверим диапазоны погодных характеристик и спроса. Это помогает понять масштаб целевой переменной ещё до обучения модели.

In [ ]:
bike[['temp', 'hum', 'wind', 'cnt']].describe().round(1)

Посмотрим, сколько наблюдений относится к каждому году. Далее будем отдельно проверять, не возникают ли систематические ошибки в одном из них.

In [ ]:
bike['year'].value_counts().sort_index().rename('Число дней')

In [ ]:
bike[['date', 'temp', 'hum', 'season', 'workingday', 'cnt']].head()

**Наблюдение.** Мы можем сопоставлять погоду и спрос, но пока не оцениваем причинный эффект погоды. Дальше будем исследовать конкретные обученные предикторы.

## 2. Как делим исторические данные

Мы исследуем, насколько обученная модель описывает **другие исторические дни** по уже измеренной погоде. Для этого целиком отложим март, июнь, сентябрь и декабрь **каждого из двух лет**. Обучение проведём на остальных месяцах: в обеих частях будут разные сезоны и оба года.

**Почему это допустимо здесь?** Мы изучаем поведение предиктора ретроспективно. Для прогноза будущих дней пришлось бы разделять данные по времени и использовать погодные прогнозы, доступные на соответствующую дату. Лагов целевой переменной среди наших признаков нет.

In [ ]:
held_out_months = [3, 6, 9, 12]
train_bike = bike.loc[~bike['month'].isin(held_out_months)].copy()
valid_bike = bike.loc[bike['month'].isin(held_out_months)].copy()

In [ ]:
pd.DataFrame({'Часть': ['Обучение', 'Оценка'], 'Дней': [len(train_bike), len(valid_bike)]})

Ожидаем 487 дней в обучении и 243 дня в оценочной части. Важно: результаты на этих же 243 днях мы позже используем для поиска недостатков модели. Если после этого изменить модель, найденное улучшение понадобится подтвердить на новых данных.

## 3. Сначала одно дерево: MDI

Начнём с трёх числовых входов: температуры, влажности и ветра. Тогда у каждого значения встроенной важности дерева будет однозначное название признака. Для регрессии дерево уменьшает **MSE в узлах**; `feature_importances_` распределяет суммарное взвешенное уменьшение по признакам.

In [ ]:
weather_cols = ['temp', 'hum', 'wind']

In [ ]:
weather_tree = DecisionTreeRegressor(max_depth=4, min_samples_leaf=15, random_state=42)

In [ ]:
weather_tree.fit(train_bike[weather_cols], train_bike['cnt']);

Проверим модель на отложенных днях. MAE измеряется в количестве прокатов за день.

In [ ]:
tree_pred = weather_tree.predict(valid_bike[weather_cols])

In [ ]:
tree_mae = mean_absolute_error(valid_bike['cnt'], tree_pred)
round(tree_mae, 1)

**Интерпретация MAE.** Например, MAE = 1200 означает, что абсолютное отклонение прогнозов от фактического числа прокатов составляет в среднем около 1200 прокатов на день. Теперь посмотрим, какие из трёх признаков дерево использовало при выборе разбиений.

In [ ]:
mdi = pd.Series(weather_tree.feature_importances_, index=weather_cols)

In [ ]:
mdi.sort_values(ascending=False).rename('MDI').to_frame().round(3)

In [ ]:
ax = mdi.sort_values().plot.barh(figsize=(8, 3.5), title='MDI дерева: доля уменьшения MSE')
ax.set_xlabel('Доля суммарного уменьшения MSE')
plt.show()

**Как читать MDI.** Сумма столбиков равна 1. Большое значение означает, что разбиения по признаку дали большую долю взвешенного уменьшения MSE **на обучении**. На этом графике видно, какие погодные признаки определили правила нашего небольшого дерева. Мы пока не знаем, насколько они важны для качества на отложенных днях: это другой вопрос.

## 4. То же дерево: permutation importance

Теперь **не меняем обученное дерево**. Для каждого входного признака несколько раз перемешаем значения столбца в оценочной таблице и посмотрим, насколько возрастёт MAE.

**Интерфейс [`permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html):**

| Аргумент | Что передаём |
|---|---|
| `estimator` | Уже обученное дерево `weather_tree` |
| `X`, `y` | Признаки и правильные ответы оценочных дней |
| `scoring` | `neg_mean_absolute_error`: больший score означает меньшую MAE |
| `n_repeats` | Сколько раз перемешать каждый столбец |
| `random_state` | Фиксирует случайные перестановки для воспроизводимости |

Результат содержит `importances_mean` и `importances_std` — средний прирост ошибки и его разброс между перестановками.

Вызов ниже использует уже обученное дерево и оценочную выборку. `scoring="neg_mean_absolute_error"` означает, что библиотека считает отрицательную MAE; положительная PI будет **приростом MAE** после перемешивания.

In [ ]:
tree_pi = permutation_importance(
    weather_tree, valid_bike[weather_cols], valid_bike['cnt'],
    scoring='neg_mean_absolute_error', n_repeats=8, random_state=42
)

In [ ]:
tree_pi_table = pd.Series(tree_pi.importances_mean, index=weather_cols, name='Прирост MAE')

In [ ]:
tree_pi_table.sort_values(ascending=False).to_frame().round(1)

**Сравниваем две величины на одном дереве.** MDI характеризует историю разбиений на обучающих строках; PI измеряет падение качества на оценочных строках при перемешивании информации. Значения PI выражены в прокатах за день и не обязаны суммироваться до 1. Для нового датасета порядок признаков может измениться.

## 5. Более богатая модель и все семь признаков

Для следующих графиков обучим случайный лес на погодных и календарных характеристиках. Он даёт более гибкую зависимость прогнозов от температуры и влажности. Мы будем использовать его через знакомые методы `fit`, `predict` и инструменты `sklearn.inspection`.

In [ ]:
bike_features = ['season', 'holiday', 'workingday', 'weather', 'temp', 'hum', 'wind']
bike_cat = ['season', 'holiday', 'workingday', 'weather']

**Как устроен `ColumnTransformer`?** Категориальные столбцы `bike_cat` превращаем в индикаторы с помощью `OneHotEncoder`. Остальные столбцы — температуру, влажность и ветер — оставляем без изменения. Именно это означает `remainder="passthrough"`. Если бы мы не указали его, `ColumnTransformer` по умолчанию удалил бы столбцы, которые не перечислены в преобразователях. [Документация](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html).

In [ ]:
bike_preprocess = ColumnTransformer(
    [('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), bike_cat)],
    remainder='passthrough'
)

In [ ]:
bike_forest = Pipeline([
    ('preprocess', bike_preprocess),
    ('forest', RandomForestRegressor(n_estimators=180, min_samples_leaf=4,
                                     max_features=0.85, random_state=42, n_jobs=-1))
])

In [ ]:
bike_forest.fit(train_bike[bike_features], train_bike['cnt']);

In [ ]:
forest_pred = bike_forest.predict(valid_bike[bike_features])

In [ ]:
forest_mae = mean_absolute_error(valid_bike['cnt'], forest_pred)
round(forest_mae, 1)

**Результат.** MAE этого леса около **1120 прокатов/день**. Модель использует семь исходных столбцов; преобразование категорий скрыто внутри Pipeline. Поэтому `permutation_importance` сможет работать с исходным DataFrame и вернёт по одной оценке на каждый исходный столбец.

### Перестановочная важность семи признаков

Тот же интерфейс работает с целым `Pipeline`. Передаём ему семь **исходных** столбцов: перестановка происходит до кодирования категорий, а все преобразования автоматически выполняются внутри `Pipeline`. Оценим изменение MAE по 12 перестановкам.

`permutation_importance` вернёт по одной важности на каждый исходный признак. В отличие от `feature_importances_` отдельные колонки после one-hot кодирования здесь не появляются: входом для PI служит весь `Pipeline` и исходный DataFrame.

In [ ]:
forest_pi = permutation_importance(
    bike_forest, valid_bike[bike_features], valid_bike['cnt'],
    scoring='neg_mean_absolute_error', n_repeats=12, random_state=42
)

In [ ]:
forest_pi_table = pd.Series(forest_pi.importances_mean, index=bike_features, name='Прирост MAE')

In [ ]:
forest_pi_table.sort_values(ascending=False).to_frame().round(1)

In [ ]:
ax = forest_pi_table.sort_values().plot.barh(figsize=(8, 4), title='PI леса на отложенных днях')
ax.set_xlabel('Прирост MAE, прокатов/день')
plt.show()

**Что показывает результат.** Сильнее всего растёт MAE при перестановке **температуры** (около +573 прокатов), затем **влажности** (около +167). Это значит, что для качества данного леса важна информация в этих признаках. Направление зависимости из PI не видно: модель может повышать или понижать прогноз в разных диапазонах температуры. Для этого построим PDP.

### Сначала посмотрим на фактические данные

Прежде чем строить PDP, рассмотрим, как в обучающей выборке связаны наблюдавшаяся температура и **фактическое** число прокатов. По горизонтали будут °C, по вертикали — спрос. Это обычная диаграмма рассеяния; её важно отличать от PDP, где по вертикали стоят усреднённые **прогнозы модели после подстановок**.

In [ ]:
ax = train_bike.plot.scatter(
    x='temp', y='cnt', alpha=0.35, figsize=(8, 4),
    title='Наблюдения: температура и число прокатов'
)
ax.set(xlabel='Температура, °C', ylabel='Фактическое число прокатов')
plt.show()

На точечной диаграмме виден разброс фактического спроса при близких температурах. У дней различаются также сезон, влажность и календарные признаки. Теперь исследуем, **как именно обученный предиктор** меняет средний прогноз при подстановке температуры.

## 6. PDP: как модель реагирует на температуру

Для каждого фиксированного значения температуры подставляем его всем оценочным дням, оставляя остальные признаки исходными. Усредняем полученные прогнозы. PDP описывает **средний прогноз модели**, а не среднее фактическое число поездок.

**Интерфейс [`PartialDependenceDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.PartialDependenceDisplay.html#sklearn.inspection.PartialDependenceDisplay.from_estimator):**

| Аргумент | Значение здесь |
|---|---|
| `estimator` | Обученный `bike_forest` (включая Pipeline) |
| `X` | Набор дней, по которым усредняем прогнозы |
| `features` | `["temp"]` — изучаем температуру |
| `kind` | `"average"` — одна кривая PDP |
| `method` | `"brute"` — делаем подстановки и вычисляем прогнозы |
| `grid_resolution` | Число значений на сетке температуры |
| `ax` | Ось Matplotlib, на которой строим график |

Сетка по умолчанию охватывает диапазон между 5-м и 95-м процентилями признака. Возвращённый объект `pdp_temp` хранит оси (`axes_`). Сразу после построения зададим их подписи и выведем готовый график.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
pdp_temp = PartialDependenceDisplay.from_estimator(
    bike_forest, valid_bike[bike_features], ['temp'],
    kind='average', method='brute', grid_resolution=45, ax=ax
)
pdp_temp.axes_[0, 0].set(
    xlabel='Подставленная температура, °C',
    ylabel='Средний прогноз, прокатов/день'
)
plt.show()


**Читаем график.** По горизонтали — подставленная температура; по вертикали — средний прогноз леса на выбранных 243 днях. У модели прогноз растёт от прохладной к тёплой погоде, затем рост замедляется и переходит в плато. Лес выучил **нелинейную форму** зависимости прогнозов от температуры. Этот график не устанавливает причинный эффект изменения погоды.

## 7. ICE: одинаково ли модель реагирует на влажность?

В PDP мы усредняли прогнозы по дням. Теперь хотим увидеть отдельные линии **до усреднения**. Для влажности построим сразу ICE и общую PDP с помощью того же `PartialDependenceDisplay`. Покажем 40 отдельных дней, чтобы линии можно было различить.

**Изменения вызова:** `features=["hum"]` выбирает влажность, `kind="both"` рисует среднюю PDP и индивидуальные ICE, `subsample=40` ограничивает только число отображаемых ICE-линий. PDP по-прежнему рассчитывается по всем 243 дням. `random_state=42` закрепляет выбор дней. У отдельных линий не будем задавать сезонные цвета: здесь наша задача — сначала увидеть неоднородность реакций.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ice_hum = PartialDependenceDisplay.from_estimator(
    bike_forest, valid_bike[bike_features], ['hum'],
    kind='both', method='brute', grid_resolution=35,
    subsample=40, random_state=42, ax=ax
)
ice_ax = ice_hum.axes_[0, 0]
ice_ax.set(xlabel='Подставленная влажность, %', ylabel='Прогноз, прокатов/день')
ice_ax.get_legend().texts[0].set_text('PDP: среднее')
plt.show()


**Читаем ICE.** Существует общий спад прогнозов при высокой влажности, однако линии начинаются с разных уровней и снижаются неодинаково. Это означает, что реакция леса на влажность зависит от остальных характеристик дня. PDP усредняет линии; ICE позволяет увидеть различия до усреднения.

### Centered ICE: сравниваем изменения, а не стартовые уровни

Обычные ICE-кривые могут начинаться на существенно разных уровнях спроса, поэтому сравнивать силу изменения неудобно. С `centered=True` библиотека вычитает из каждой кривой её прогноз при первом значении общей сетки влажности. **Все линии начинают с нуля; различия между двумя точками одной линии сохраняются.**

Остальные параметры оставим теми же: `kind="both"`, `subsample=40`, `random_state=42`. [Документация `PartialDependenceDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.PartialDependenceDisplay.html#sklearn.inspection.PartialDependenceDisplay.from_estimator).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ice_centered = PartialDependenceDisplay.from_estimator(
    bike_forest, valid_bike[bike_features], ['hum'],
    kind='both', centered=True, method='brute',
    grid_resolution=35, subsample=40, random_state=42, ax=ax
)
center_ax = ice_centered.axes_[0, 0]
center_ax.set(
    xlabel='Подставленная влажность, %',
    ylabel='Изменение прогноза, прокатов/день'
)
center_ax.get_legend().texts[0].set_text('PDP: среднее')
plt.show()


**Интерпретация.** Теперь по вертикали отложено изменение прогнозов от общей стартовой влажности. Видно, что в области высокой влажности одни линии падают заметнее других. Центрирование не меняет обученную модель и не изменяет разности между точками одной ICE-кривой — оно лишь убирает различие начальных уровней.

## 8. Диагностика ошибок: средняя MAE скрывает различия

До сих пор мы исследовали, как **модель меняет прогнозы**. Теперь сопоставим их с фактическим спросом. Остаток определим как `факт − прогноз`: положительный остаток означает недооценку, отрицательный — завышение.

In [ ]:
bike_errors = valid_bike[['date', 'year', 'cnt']].copy()
bike_errors['prediction'] = forest_pred

In [ ]:
bike_errors['residual'] = bike_errors['cnt'] - bike_errors['prediction']
bike_errors['absolute_error'] = bike_errors['residual'].abs()

In [ ]:
bike_errors.head()

Посмотрим среднюю абсолютную ошибку (MAE) и средний **знаковый** остаток (ME) отдельно по годам. Эти показатели отвечают на разные вопросы: насколько велики ошибки и в какую сторону модель чаще ошибается.

In [ ]:
bike_year_errors = bike_errors.groupby('year').agg(
    n=('cnt', 'size'), MAE=('absolute_error', 'mean'), ME=('residual', 'mean')
)

In [ ]:
bike_year_errors.round(1)

In [ ]:
ax = bike_year_errors['ME'].plot.bar(figsize=(7, 3.5), title='Средний остаток по годам')
ax.axhline(0, color='gray', linewidth=1)
ax.set_ylabel('ME = факт − прогноз, прокатов/день')
ax.set_xlabel('Год')
ax.tick_params(axis='x', rotation=0)
plt.show()

**Что обнаружили.** В 2011 году ME отрицателен (порядка −911): модель чаще завышает спрос. В 2012 году ME положителен (около +1171): модель чаще занижает. Общая MAE ≈1120 не показывает этой противоположной направленности. Возможная причина — модель не различает два исторических уровня спроса, поскольку **год отсутствует среди входных признаков**.

### Проверка одной гипотезы

Добавим год (2011/2012) в список признаков и обучим **новый лес с теми же настройками**. Сравним MAE на прежних оценочных месяцах. В отличие от PI, PDP и ICE здесь мы действительно переобучаем модель.

In [ ]:
bike_features_year = bike_features + ['year']

In [ ]:
forest_with_year = Pipeline([
    ('preprocess', ColumnTransformer(
        [('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), bike_cat)],
        remainder='passthrough'
    )),
    ('forest', RandomForestRegressor(n_estimators=180, min_samples_leaf=4,
                                     max_features=0.85, random_state=42, n_jobs=-1))
])

In [ ]:
forest_with_year.fit(train_bike[bike_features_year], train_bike['cnt']);

In [ ]:
year_pred = forest_with_year.predict(valid_bike[bike_features_year])
year_mae = mean_absolute_error(valid_bike['cnt'], year_pred)

In [ ]:
pd.DataFrame({'Без года': [forest_mae], 'С годом': [year_mae]}, index=['MAE, прокатов/день']).round(1)

**Вывод исследования.** Добавление года уменьшило MAE примерно с **1120 до 569 прокатов/день** на этих отложенных месяцах. Мы нашли содержательный недостаток модели и проверили одно изменение. Однако гипотезу предложили после изучения тех же 243 дней, на которых сравнили модели. Для подтверждения результата нужна новая независимая оценка; прогноз спроса на 2013 год в этом эксперименте не проверялся.

---

# Часть II. Бриллианты: самостоятельное исследование модели

Теперь исследуем **другую задачу регрессии**, используя те же библиотечные интерфейсы. Основной технический код подготовлен. **Ваша работа** — выполнить ячейки, разобраться в результатах, письменно ответить на четыре вопроса в блоках «Ваш вывод» и сделать две небольшие модификации параметров графиков. Дополнительный эксперимент с размерами камня выполняется по желанию.

Данные [Diamonds](https://ggplot2.tidyverse.org/reference/diamonds.html) входят в пакет `plotnine`. Исходная таблица содержит исторические предложения о продаже бриллиантов; **одна строка — один камень**, целевая переменная `price` — цена в долларах США. Отдельный CSV не потребуется.

## 9. Сначала посмотрим данные

Подключим таблицу из установленной библиотеки и посмотрим первые строки.

In [ ]:
from plotnine.data import diamonds

In [ ]:
gems = diamonds.copy()

In [ ]:
gems.head()

Основные признаки:

- `carat` — масса в каратах;
- `cut` — качество огранки;
- `color` — категория цвета;
- `clarity` — категория чистоты;
- `depth` и `table` — геометрические характеристики огранки;
- `x`, `y`, `z` — размеры камня в миллиметрах (используем в дополнительном сравнении).

`price` — фактическая цена в долларах. Внимательно прочитайте, какие признаки числовые, а какие категориальные.

In [ ]:
gems.shape

In [ ]:
gems[['carat', 'price', 'depth', 'table']].describe().round(1)

Подготовим **воспроизводимую подвыборку из 5000 камней**. Удаляем точные дубликаты и строки с нулевыми или неправдоподобно большими линейными размерами. Это позволяет одинаково воспроизвести таблицы и сравнения.

In [ ]:
gems = gems.drop_duplicates().copy()

In [ ]:
plausible_sizes = ((gems[['x', 'y', 'z']] > 0) & (gems[['x', 'y', 'z']] < 15)).all(axis=1)
gems = gems.loc[plausible_sizes]

In [ ]:
gems = gems.sample(n=5000, random_state=42).reset_index(drop=True)

In [ ]:
for column in ['cut', 'color', 'clarity']:
    gems[column] = gems[column].astype(str)

In [ ]:
gems[['carat', 'cut', 'color', 'clarity', 'price']].head()

**Первое наблюдение.** Масса и цена имеют разный масштаб: более крупные камни нередко стоят существенно дороже. Но мы пока не знаем, какую связь выучит дерево и насколько его качество зависит от каждого входа. Это и проверим.

## 10. Обучим дерево

Используем шесть признаков. Категории внутри `Pipeline` преобразуются в числовые индикаторы. Подбор гиперпараметров здесь не выполняем: настройки дерева заранее фиксированы, чтобы сосредоточиться на интерпретации.

In [ ]:
gem_features = ['carat', 'cut', 'color', 'clarity', 'depth', 'table']
gem_categorical = ['cut', 'color', 'clarity']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor

In [ ]:
gem_train_idx, gem_valid_idx = train_test_split(
    np.arange(len(gems)), test_size=0.27, random_state=42
)

In [ ]:
X_gem_train = gems.loc[gem_train_idx, gem_features]
X_gem_valid = gems.loc[gem_valid_idx, gem_features]

In [ ]:
y_gem_train = gems.loc[gem_train_idx, 'price']
y_gem_valid = gems.loc[gem_valid_idx, 'price']

Числовые признаки бриллиантов оставим в исходных единицах, а `cut`, `color`, `clarity` перекодируем в индикаторы. `remainder="passthrough"` сохраняет все остальные столбцы, в том числе `carat`, `depth` и `table`; по умолчанию они были бы исключены.

In [ ]:
gem_preprocess = ColumnTransformer(
    [('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), gem_categorical)],
    remainder='passthrough'
)

In [ ]:
gem_tree = Pipeline([
    ('preprocess', gem_preprocess),
    ('tree', DecisionTreeRegressor(max_depth=8, min_samples_leaf=22, random_state=42))
])

In [ ]:
gem_tree.fit(X_gem_train, y_gem_train);

Сначала оценим MAE, затем сравним с простым прогнозом «всем камням одинаковую медианную цену обучающей выборки». Такое сравнение помогает понять масштаб ошибки дерева.

In [ ]:
gem_pred = gem_tree.predict(X_gem_valid)

In [ ]:
gem_mae = mean_absolute_error(y_gem_valid, gem_pred)
round(gem_mae, 1)

In [ ]:
gem_baseline = DummyRegressor(strategy='median').fit(X_gem_train, y_gem_train);

In [ ]:
baseline_mae = mean_absolute_error(y_gem_valid, gem_baseline.predict(X_gem_valid))
round(baseline_mae, 1)

**Как читать результат.** MAE измеряется в долларах за камень. Если дерево имеет MAE ниже константной модели, оно действительно использует доступные характеристики для более точных прогнозов на нашей оценочной выборке. Разница в MAE сама по себе ещё не раскрывает, какие признаки обеспечили результат.

## 11. Permutation importance

Для того же обученного дерева перемешаем исходные столбцы **оценочной** таблицы. Благодаря `Pipeline` имена в результате будут относиться к исходным признакам, даже если `cut`, `color` и `clarity` преобразованы в несколько индикаторов внутри модели.

**Какие аргументы нужны?** `estimator=gem_tree` — уже обученный Pipeline; `X_gem_valid` и `y_gem_valid` — оценочные данные; `scoring` выбирает MAE со знаком минус; `n_repeats=7` управляет числом перемешиваний. Используем тот же интерфейс, который только что разобрали на велосипедах.

In [ ]:
gem_pi = permutation_importance(
    gem_tree, X_gem_valid, y_gem_valid,
    scoring='neg_mean_absolute_error', n_repeats=7, random_state=42
)

In [ ]:
gem_pi_values = pd.Series(gem_pi.importances_mean, index=gem_features, name='Прирост MAE, $')

In [ ]:
gem_pi_values.sort_values(ascending=False).to_frame().round(1)

In [ ]:
ax = gem_pi_values.sort_values().plot.barh(figsize=(8, 4), title='Перестановочная важность признаков')
ax.set_xlabel('Прирост MAE после перестановки, $')
plt.show()

**На что смотреть.** Самый длинный столбик соответствует признаку, перемешивание которого сильнее всего ухудшило прогнозы *этого дерева*. Положительная PI в долларах означает прирост ошибки после перестановки. Небольшой столбик не доказывает, что характеристика бесполезна для любой модели или всех сделок с бриллиантами.

**Ваш вывод 1.** Какой признак получил максимальную PI? Насколько выросла MAE при его перестановке? Что это говорит о зависимости **именно нашего дерева** от этого признака?

**Ваш ответ:** _Запишите величину PI в долларах и объясните её смысл для этой модели._

**Интерфейс PDP тот же.** Теперь `features=["carat"]`: вместо температуры исследуем массу. `kind="average"` усреднит прогнозы по всем оценочным камням; `grid_resolution=45` задаёт количество точек сетки. У обученного дерева кривая может быть ступенчатой.

## 12. PDP: форма зависимости прогноза от массы

Теперь исследуем **как** дерево использует `carat`. Для каждой точки сетки подставим всем оценочным камням одну и ту же массу, оставляя остальные характеристики, и усредним прогнозы. Получится PDP.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
gem_pdp = PartialDependenceDisplay.from_estimator(
    gem_tree, X_gem_valid, ['carat'],
    kind='average', method='brute', grid_resolution=45, ax=ax
)
gem_pdp.axes_[0, 0].set(
    xlabel='Подставленная масса, карат', ylabel='Средний прогноз цены, $'
)
plt.show()


**Как читать PDP.** Горизонтальная ось — подставленная масса, вертикальная — средний прогноз дерева по оценочным камням. У дерева кривая часто **ступенчатая**, поскольку оно разделяет диапазоны признаков порогами. Оцените, в каких диапазонах прогноз меняется особенно резко.

**Ваш вывод 2.** Как изменяется средний прогноз цены с ростом массы? Есть ли пороги или участки почти постоянного прогноза?

**Ваш ответ:** _Опишите форму и пороги, которые видны на графике._

**Задание с кодом 1.** Вернитесь в ячейку построения PDP для `carat` и временно замените `features=["carat"]` на `features=["depth"]`. Сравните, насколько заметно меняется прогноз по глубине огранки. Затем верните `carat`, чтобы продолжить с ICE. Достаточно изменить одно имя признака.

**Интерфейс ICE тот же.** Меняем `kind` на `"both"`, чтобы увидеть и среднюю PDP, и 40 индивидуальных кривых (`subsample=40`). Сможем сравнить, одинаково ли дерево реагирует на изменение массы у камней с разными другими характеристиками.

## 13. ICE: сравним отдельные камни

PDP усредняет прогнозы. ICE сохраняет кривую **для каждого камня** до усреднения. Чтобы график можно было прочитать, покажем 40 отдельных линий на фоне одной общей PDP.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
gem_ice = PartialDependenceDisplay.from_estimator(
    gem_tree, X_gem_valid, ['carat'],
    kind='both', method='brute', grid_resolution=35,
    subsample=40, random_state=42, ax=ax
)
gem_ice_ax = gem_ice.axes_[0, 0]
gem_ice_ax.set(xlabel='Подставленная масса, карат', ylabel='Прогноз цены, $')
gem_ice_ax.get_legend().texts[0].set_text('PDP: среднее')
plt.show()


**Как читать ICE.** Каждая тонкая линия принадлежит отдельному камню с фиксированными `cut`, `color`, `clarity`, `depth`, `table`. Если линии отличаются по высоте или ступеням, дерево реагирует на одинаковую массу по-разному в зависимости от остальных характеристик. Общая PDP сглаживает эти различия.

**Ваш вывод 3.** Что видно по ICE сверх того, что уже показала PDP? Как связаны различия индивидуальных линий с остальными характеристиками камней?

**Ваш ответ:** _Назовите одно различие между PDP и индивидуальными ICE-кривыми._

**Задание с кодом 2.** Попробуйте заменить `subsample=40` на `subsample=100` и повторить построение ICE. Отметьте, легче или труднее стало различать отдельные линии. Верните 40 для последующего чтения графиков.

## 14. Диагностика ошибок по массе

Теперь сопоставим **фактические цены** с прогнозами. Разделим оценочные камни по *наблюдаемой массе* — признаку, который доступен ещё до прогноза. Для каждой группы посчитаем количество объектов и MAE в долларах.

In [ ]:
gem_errors = gems.loc[gem_valid_idx, ['carat', 'price']].copy()
gem_errors['prediction'] = gem_pred

In [ ]:
gem_errors['absolute_error'] = (gem_errors['price'] - gem_errors['prediction']).abs()

In [ ]:
gem_errors.head()

In [ ]:
gem_errors['mass_group'] = pd.cut(
    gem_errors['carat'], bins=[0, 0.5, 1, 1.5, 6],
    labels=['до 0,5', '0,5–1', '1–1,5', 'от 1,5']
)

In [ ]:
gem_group_mae = gem_errors.groupby('mass_group', observed=True)['absolute_error'].agg(n='size', MAE='mean')

In [ ]:
gem_group_mae.round(1)

In [ ]:
ax = gem_group_mae['MAE'].plot.bar(figsize=(7, 3.5), title='MAE по диапазонам массы')
ax.set(xlabel='Масса бриллианта, карат', ylabel='MAE, $')
plt.show()

**Как читать таблицу и график.** Крупные камни обычно дороже, поэтому ошибка в долларах может быть выше при том же относительном отклонении. Сравнивайте и MAE, и число наблюдений в группе. Значительно меньшая группа даёт менее устойчивую оценку. Если нужно проверить именно относительную точность, отдельно рассчитайте процентные или нормированные ошибки.

**Ваш вывод 4.** В каком диапазоне массы MAE выше? Насколько отличается число камней между группами? Какую следующую проверку вы предложили бы после этого наблюдения?

**Ваш ответ:** _Назовите группу, сравните её MAE с другой и предложите следующую проверку._

## 15. Дополнительно: связанные признаки и PI

Кроме массы, у камня есть размеры `x`, `y`, `z` в миллиметрах. Они очень тесно связаны с `carat`. Если обучить новое дерево на расширенном наборе признаков, оно может использовать информацию о размере **через другие столбцы**.

Здесь мы действительно заново обучаем модель с дополнительными входами. Не путайте этот эксперимент с перестановкой столбца в фиксированной модели.

In [ ]:
gem_extended_features = gem_features + ['x', 'y', 'z']

In [ ]:
gems[['carat', 'x', 'y', 'z']].corr().round(2)

In [ ]:
gem_extended_preprocess = ColumnTransformer(
    [('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), gem_categorical)],
    remainder='passthrough'
)

In [ ]:
gem_extended_tree = Pipeline([
    ('preprocess', gem_extended_preprocess),
    ('tree', DecisionTreeRegressor(max_depth=8, min_samples_leaf=22, random_state=42))
])

In [ ]:
gem_extended_tree.fit(gems.loc[gem_train_idx, gem_extended_features], y_gem_train);

In [ ]:
gem_extended_pred = gem_extended_tree.predict(gems.loc[gem_valid_idx, gem_extended_features])

In [ ]:
round(mean_absolute_error(y_gem_valid, gem_extended_pred), 1)

In [ ]:
gem_extended_pi = permutation_importance(
    gem_extended_tree, gems.loc[gem_valid_idx, gem_extended_features], y_gem_valid,
    scoring='neg_mean_absolute_error', n_repeats=7, random_state=42
)

In [ ]:
gem_extended_pi_values = pd.Series(
    gem_extended_pi.importances_mean, index=gem_extended_features, name='Прирост MAE, $'
)

In [ ]:
pd.DataFrame({'Исходное дерево': gem_pi_values, 'С размерами x/y/z': gem_extended_pi_values}).round(1)

**Что сравниваем.** Состав признаков и обученное дерево изменились, а разбиение данных осталось прежним. Из-за высокой корреляции масса может потерять часть PI: новая модель использует связанные геометрические признаки. Оценивать полезность признака для **всех возможных моделей** по PI одной конкретной модели нельзя.

**Дополнительный вывод.** Что произошло с PI `carat` после добавления размеров? Какой признак стал особенно важен? Улучшилась ли MAE после переобучения?

**Дополнительный ответ:** _Что изменилось после добавления размеров? Улучшилось ли качество?_

---

## Итак: что удалось исследовать

- **MDI:** какие признаки дерево использовало при обучающих разбиениях.
- **PI:** от каких входных столбцов зависит качество фиксированного предиктора.
- **PDP:** как изменяется средний прогноз при одинаковой подстановке значения признака.
- **ICE:** как реакция различается для отдельных объектов.
- **Диагностика:** где ошибки больше и какую проверяемую гипотезу можно предложить.

В обоих примерах мы исследовали **обученные модели на конкретных выборках**. Для подтверждения найденного улучшения после изменения модели нужны новые оценочные данные.

**Источники:** [UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) (CC BY 4.0); [Diamonds в ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html) и [plotnine](https://plotnine.org/reference/diamonds.html); API [permutation_importance](https://scikit-learn.org/stable/modules/permutation_importance.html) и [PartialDependenceDisplay](https://scikit-learn.org/stable/modules/partial_dependence.html).